<a href="https://colab.research.google.com/github/tomideleshi/retail-sales-etl-pipeline/blob/main/retail_sales_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")


Libraries imported successfully!


In [12]:
# EXTRACT - Create raw messy sales data
data = {
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'customer_name': ['Alice Johnson', 'bob smith', 'CAROL WHITE', 'David Brown', 'alice johnson', 'Frank Miller', 'grace wilson', 'Henry Moore', None, 'Frank Miller'],
    'product': ['Laptop', 'Phone', 'Tablet', 'laptop', 'PHONE', 'Monitor', 'Tablet', 'Keyboard', 'Laptop', None],
    'quantity': [2, 1, 3, 1, 2, None, 1, 3, 2, 1],
    'price': [799.99, 499.99, 599.99, 799.99, 499.99, 399.99, 599.99, 79.99, 799.99, 399.99],
    'order_date': ['2024-01-05', '2024-01-08', '2024-01-12', '2024-01-15', '2024-01-18', '2024-01-20', '2024-01-22', '2024-01-25', '2024-02-01', '2024-02-05']
}

df_raw = pd.DataFrame(data)
print("Raw data extracted!")
print(df_raw)

Raw data extracted!
   order_id  customer_name   product  quantity   price  order_date
0         1  Alice Johnson    Laptop       2.0  799.99  2024-01-05
1         2      bob smith     Phone       1.0  499.99  2024-01-08
2         3    CAROL WHITE    Tablet       3.0  599.99  2024-01-12
3         4    David Brown    laptop       1.0  799.99  2024-01-15
4         5  alice johnson     PHONE       2.0  499.99  2024-01-18
5         6   Frank Miller   Monitor       NaN  399.99  2024-01-20
6         7   grace wilson    Tablet       1.0  599.99  2024-01-22
7         8    Henry Moore  Keyboard       3.0   79.99  2024-01-25
8         9           None    Laptop       2.0  799.99  2024-02-01
9        10   Frank Miller      None       1.0  399.99  2024-02-05


In [13]:
# TRANSFORM - Clean the messy data
df_clean = df_raw.copy()

# Fix capitalization on customer names and products
df_clean['customer_name'] = df_clean['customer_name'].str.title()
df_clean['product'] = df_clean['product'].str.title()

# Fill missing quantity with average quantity
df_clean['quantity'] = df_clean['quantity'].fillna(df_clean['quantity'].mean())

# Drop rows where customer name or product is missing
df_clean = df_clean.dropna(subset=['customer_name', 'product'])

# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['customer_name', 'product'])

# Add a total sales column
df_clean['total_sales'] = df_clean['quantity'] * df_clean['price']

print("Data transformed successfully!")
print(df_clean)

Data transformed successfully!
   order_id  customer_name   product  quantity   price  order_date  \
0         1  Alice Johnson    Laptop  2.000000  799.99  2024-01-05   
1         2      Bob Smith     Phone  1.000000  499.99  2024-01-08   
2         3    Carol White    Tablet  3.000000  599.99  2024-01-12   
3         4    David Brown    Laptop  1.000000  799.99  2024-01-15   
4         5  Alice Johnson     Phone  2.000000  499.99  2024-01-18   
5         6   Frank Miller   Monitor  1.777778  399.99  2024-01-20   
6         7   Grace Wilson    Tablet  1.000000  599.99  2024-01-22   
7         8    Henry Moore  Keyboard  3.000000   79.99  2024-01-25   

   total_sales  
0  1599.980000  
1   499.990000  
2  1799.970000  
3   799.990000  
4   999.980000  
5   711.093333  
6   599.990000  
7   239.970000  


In [14]:
# LOAD - Save cleaned data into a SQLite database
conn = sqlite3.connect('sales_data.db')

df_clean.to_sql('clean_sales', conn, if_exists='replace', index=False)

print("Data loaded into database successfully!")

# Verify it worked by reading it back
df_verify = pd.read_sql("SELECT * FROM clean_sales", conn)
print(df_verify)

conn.close()

Data loaded into database successfully!
   order_id  customer_name   product  quantity   price  order_date  \
0         1  Alice Johnson    Laptop  2.000000  799.99  2024-01-05   
1         2      Bob Smith     Phone  1.000000  499.99  2024-01-08   
2         3    Carol White    Tablet  3.000000  599.99  2024-01-12   
3         4    David Brown    Laptop  1.000000  799.99  2024-01-15   
4         5  Alice Johnson     Phone  2.000000  499.99  2024-01-18   
5         6   Frank Miller   Monitor  1.777778  399.99  2024-01-20   
6         7   Grace Wilson    Tablet  1.000000  599.99  2024-01-22   
7         8    Henry Moore  Keyboard  3.000000   79.99  2024-01-25   

   total_sales  
0  1599.980000  
1   499.990000  
2  1799.970000  
3   799.990000  
4   999.980000  
5   711.093333  
6   599.990000  
7   239.970000  


In [15]:
# Pipeline Summary
print("ETL PIPELINE SUMMARY")
print("====================")
print(f"Raw records extracted: {len(df_raw)}")
print(f"Clean records loaded: {len(df_clean)}")
print(f"Records removed: {len(df_raw) - len(df_clean)}")
print(f"Total sales value loaded: ${df_clean['total_sales'].sum():,.2f}")
print("Pipeline completed successfully!")

ETL PIPELINE SUMMARY
Raw records extracted: 10
Clean records loaded: 8
Records removed: 2
Total sales value loaded: $7,250.96
Pipeline completed successfully!
